#### Pipeline
- 여러 단계(전처리, 변환, 추정)를 연속적으로 연결하여 실행하는 도구
- 전처리 과정과 학습 모델을 하나의 객체(class)로 결합하여 사용
- 교차검증(kfold), 파라미터 탐색(GridSearch)에서 유용
- 데이터의 누수로 방지 : scaler를 사용하는 경우에는 train data만 자동으로 fit하고 test는 transform
- parameter
    - steps
        - 필수 항목 (기본값이 존재하지 않는다)
        - 파이프 라인의 단계들을 별칭과 객체들을 쌍으로 묶어서 list 형태로 구성
        - ex) [('scaler', StandardScaler()), ('svm',SVC())]
    - verbase
        - 기본값 : False
        - 각 스텝이 실행이 될때 로그를 출력할것인가? (진행 상황을 표기)
- 속성
    - name_steps
        - 파이프라인의 각 단계를 딕셔너리형처럼 접근 하는 방법
- 메서드
    - fit( X, y, fit_params )
        - 순서대로 각 단계의 fit() 실행
    - fit_transfomr(X, y, fir_params)
        - 순서대로 각 단계의 transform()을 진행
    - predict(X)
        - 마지막 단계에서 만들어진 모델에 학습

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
iris = pd.read_csv('../data/iris.csv')
iris.head()

,sepal length,sepal width,petal length,petal width,target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
from sklearn.preprocessing import LabelEncoder

In [4]:
le = LabelEncoder()

iris['target'] = le.fit_transform(iris['target'])
iris['target'].value_counts()

target
0    50
1    50
2    50
Name: count, dtype: int64

In [5]:
# train, test 데이터셋 qnsgkf
x = iris.drop('target', axis = 1)
y = iris['target']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# 파이프라인 생성 -> 객체 생성
pipe = Pipeline(
    [
        ('scaler', StandardScaler()),   # 1 step
        ('model', SVC() )               # 2 step
    ]
)


In [7]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


In [8]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


In [9]:
pred = pipe.predict(X_test)

In [10]:
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



#### GridSearchCV
- 하이퍼 파라미터(매개변수) 조합을 탐색하여 최적의 조합을 찾는 방법

- 각 파라미터 조합별로 교차검증(CV)을 수행해 평균 성능을 비교

- 최적의 모델과 성능을 자동으로 제공

- parameter

    - estimator
        - 모델의 선택
        - 학습 모델, Pipeline
        -모델을 선택할때 고정 파라미터는 모델선택과정에서 지정
    - param_grid
        - 탐색할 파라미터의 조합 (dict 형태로)
        - ex
            - 학습 모델을 선택 : { 'C' : [0.1, 1, 10], 'kernel' : ['linear', 'rbf'] }
            - Pipeline을 선택 : { 'svm__C' : [0.1, 1, 10], 'svm__kernel' : ['linear', 'rbf'] }
    - scoring
        - 기본값 : None
        - 평가 지표 설정
        - 회귀 모델에 성능 좋은 모델을 찾기 위해서는 neg_mse | mae (오차에 음수형으로 점수를 바꿔주는 작업)
    - cv
        - 기본값 : None
        - 교차 검증의 횟수
        - 5 -> K-Fold(5)
        - train, test 데이터셋을 5회 생성 (회귀에서는 랜덤하게 폴드화 진행)
        - 분류 모델에서는 StratifiedKFold를 이용(자동 전환) -> 계층화 폴드 작업
        - Kfold() 입력값으로 사용 가능
    - refit
        - 기본값 : True
        - 최적의 파라미터로 전체 데이터를 재 학습할 것인가?
        - True 사용 시 속성에서 best_estimator_을 이용하여 모델을 로드
        - False 사용 시 속성에서 best_params_를 이용하여 최적의 파라미터를 출력
    - return_train_score
        - 기본값 : False
        - 교차 검증 시 훈련 성능 점수까지 반영 할것인가?
        - ex
            - train_score // test_score
            - 두개의 점수가 모두 높다면 -> Best
            - train점수는 높고 test의 점수가 낮은 경우 -> 과적합
            - 두개의 점수가 모두 낮다면 -> 데이터 자체에서 문제이거나 추가적인 데이터 튜닝 필요
    - n_jobs
        - 기본값 : None
        - 병렬 처리할 CPU의 개수를 지정 (-1이면 모든 CPU를 사용)
    - verbose
        - 기본값 : 0
        - 0 : 모든 로그를 표시x
        - 1 : 로그를 축약해서 표시 (간단하게 보기)
        - 2 : 풀 로그 출력 (상세하게 보기)
    - error_score
        - 기본값 : np.nan(결측치)
        - 모델 학습 시 에러가 발생할때 점수를 어떻게 표현할것인가?(점수에 대한 기본값)

- 속성

    - cv_results
        - 각 파라미터 조합별 성능의 결과 (훈련 / 검증 검수, fit 시간)
    - best_estimator_
        - 최적의 파라미터로 재 학습이 된 모델 객체
    - bset_parmas_
        - 최적의 성능을 낸 파라미터들의 조합
    - best_score_
        - 최적의 파라미터로 조합이 된 모델의 교차 검증 평균 성능
    - refit_time_
        - 재 학습에 걸린 시간
- 메서드

    - fit()
        - 하이퍼 파라미터 조합별 교차 검증 시작
    - predict()
        - 최적의 모델을 이용하여 예측
    - predict_proba()
        - 분류인 경우 확률 예측
    - score
        - 최적의 모델의 점수를 출력

In [11]:
from sklearn.model_selection import GridSearchCV

In [12]:
# 검증
# SVC 모델을 생성 
svc = SVC()
# svc에서 사용할 매개변수의 목록들을 작성 
grid_dict = {
    'C' : [0.1, 1, 10], 
    'kernel' : ['linear', 'rbf'], 
    'gamma' : ['scale', 'auto']
}

grid = GridSearchCV(
    estimator= svc, 
    param_grid= grid_dict, 
    cv = 5, 
    scoring = 'accuracy',
    verbose=1, 
    refit = True

)

In [13]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3

In [14]:
print("최적의 파라미터 조합 : ", grid.best_params_)
print("최적의 스코어 : ", grid.best_score_)
print("최적의 모델 : ", grid.best_estimator_)

최적의 파라미터 조합 :  {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
최적의 스코어 :  0.9833333333333334
최적의 모델 :  SVC(C=0.1, kernel='linear')


#### KFold
- 데이터셋을 K개의 동일 크기 부분(Fold)으로 나눠서 K번 반복하여 하나의 폴드를 검증용으로 사용을 하고 나머지 K-1개의 폴드를 학습용으로 사용

- 모든 폴드를 한번씩은 검증용으로 포함을 시켜서 성능을 안정적으로 평가 할 수 있는 방법

- 데이터의 개수가 적은 경우 반복 실행 작업을 이용하여 성능을 안정적으로 평가

- parameter

    - n_splits
        - 기본값 : 5
        - 폴드의 개수를 지정
        - 최소 값은 2
    - shuffle
        - 기본값 : False
        - 데이터를 분할하기 전에 데이터들을 섞을지 지정
        - True로 변경하면 폴드 안에 데이터가 랜덤하게 구성
    - random_state
        - 기본값 : None
        - shuffle이 True인 경우에만 사용
        - 랜덤 시드를 고정
- 속성

    - n_splits
        - 분할된 폴드의 개수

- 메서드

    - split(X, y = None)
        - 학습용 검증용 인덱스를 생성
        - 반복문을 이용하여 (train_index, test_index)로 변환하여 사용
- 장점

    - 데이터를 폴드화해서 학습/검증을 사용하기 때문에 데이터의 낭비가 없다.
    - train_test_split에 비해서 성능 평가 안정적
- 단점

    - K번의 학습 -> K번의 예측 -> K번의 평가 : 계산 횟수가 늘어난다. 시간이 증가
    - 데이터의 규모가 커지면 커질수록 시간은 크게 증가
- 특수한 경우 변형 KFold 종류

    - StratifiedKFold : 분류 문제에거 클래스의 비율을 유지하며 분할
    - GroupKFold : 그룹 단위로 데이터를 나눠 특정 그룹으로 학습하고 다른 그룹으로 검증을 하는 방법
    - RepeatedKFold : KFold를 여러번 반복해 평가 안정성 강화

In [15]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

In [16]:
X = np.array(
    ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
)

# KFold 
kfold = KFold( n_splits=2, shuffle=True, random_state=42)

# list(kfold.split(X))

for train_idx, test_idx in kfold.split(X):
    # print(train_idx)
    print('-' * 60)
    print("학습 데이터의 목록 : ", X[train_idx])
    print('검증 데이터의 목록 : ', X[test_idx])

------------------------------------------------------------
학습 데이터의 목록 :  ['C' 'D' 'E' 'G' 'J']
검증 데이터의 목록 :  ['A' 'B' 'F' 'H' 'I']
------------------------------------------------------------
학습 데이터의 목록 :  ['A' 'B' 'F' 'H' 'I']
검증 데이터의 목록 :  ['C' 'D' 'E' 'G' 'J']


In [17]:
y = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
# list(kfold.split(X, y))
for train_idx, test_idx in kfold.split(X):
    # print(train_idx)
    print('-' * 60)
    print("학습 데이터의 목록 : ", X[train_idx])
    print("학습 정답의 목록 : ", y[train_idx])
    print('검증 데이터의 목록 : ', X[test_idx])
    print('검증 정답의 목록 : ', y[test_idx])

------------------------------------------------------------
학습 데이터의 목록 :  ['C' 'D' 'E' 'G' 'J']
학습 정답의 목록 :  [2 3 4 6 9]
검증 데이터의 목록 :  ['A' 'B' 'F' 'H' 'I']
검증 정답의 목록 :  [0 1 5 7 8]
------------------------------------------------------------
학습 데이터의 목록 :  ['A' 'B' 'F' 'H' 'I']
학습 정답의 목록 :  [0 1 5 7 8]
검증 데이터의 목록 :  ['C' 'D' 'E' 'G' 'J']
검증 정답의 목록 :  [2 3 4 6 9]


In [18]:

from sklearn.svm import SVR

In [19]:
x = iris.drop('target', axis=1)
y = iris['target']

In [20]:
# Pipeline + GridSearchCV + StratifiedKFold 
cv_fold = StratifiedKFold(shuffle=True, random_state=42)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
# Pipeline 생성 
pipe = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('clf', SVC(random_state=42, probability=True))
    ]
)

In [23]:
# clf에서 사용할 파라미터 조합 생성 
# GridSearchCV에서 사용할 파라미터 조합 ( 별칭__매개변수명 )
params = {
    'clf__C' : [0.1, 1, 10], 
    'clf__gamma' : ['scale', 'auto'], 
    'clf__kernel' : ['linear', 'rbf']
}

In [24]:
grid_clf = GridSearchCV(
    estimator= pipe,        # 파이프라인으로 생성한 모델을 지정
    param_grid= params,     # dict 형태로 파라미터의 조합 
    scoring= 'accuracy',    # 베스트 모델 선정 기준
    cv = cv_fold,           # 생성해둔 KFold 지정
    refit = True,           # 재학습 여부
    n_jobs = -1,            # 모든 CPU를 사용
    return_train_score= True,   # 학습 데이터의 성능을 출력
    verbose= 2              # 로그 표시를 상세하게 
)

In [25]:

grid_clf.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__C': [0.1, 1, ...], 'clf__gamma': ['scale', 'auto'], 'clf__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and param

In [26]:

grid_clf.score(X_test, y_test)

0.9333333333333333

In [27]:
grid_df = pd.DataFrame(grid_clf.cv_results_)
grid_df.sort_values('mean_test_score', ascending=False)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__C,param_clf__gamma,param_clf__kernel,params,split0_test_score,split1_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.017251,0.009473,0.005504,0.001846,0.1,scale,linear,"{'clf__C': 0.1, 'clf__gamma': 'scale', 'clf__k...",0.958333,1.000000,...,0.975000,0.020412,1,0.968750,0.968750,0.979167,0.979167,0.968750,0.972917,0.005103
2,0.009508,0.001770,0.003522,0.000650,0.1,auto,linear,"{'clf__C': 0.1, 'clf__gamma': 'auto', 'clf__ke...",0.958333,1.000000,...,0.975000,0.020412,1,0.968750,0.968750,0.979167,0.979167,0.968750,0.972917,0.005103
4,0.007915,0.000752,0.009691,0.011345,1.0,scale,linear,"{'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...",1.000000,1.000000,...,0.975000,0.033333,1,0.968750,0.968750,0.979167,0.979167,0.989583,0.977083,0.007795
6,0.008241,0.001703,0.004578,0.001183,1.0,auto,linear,"{'clf__C': 1, 'clf__gamma': 'auto', 'clf__kern...",1.000000,1.000000,...,0.975000,0.033333,1,0.968750,0.968750,0.979167,0.979167,0.989583,0.977083,0.007795
5,0.010578,0.002893,0.004735,0.001532,1.0,scale,rbf,"{'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...",0.958333,1.000000,...,0.966667,0.016667,5,0.979167,0.979167,0.968750,0.979167,0.979167,0.977083,0.004167
7,0.009343,0.001823,0.003725,0.000510,1.0,auto,rbf,"{'clf__C': 1, 'clf__gamma': 'auto', 'clf__kern...",0.958333,1.000000,...,0.966667,0.016667,5,0.979167,0.979167,0.968750,0.979167,0.979167,0.977083,0.004167
8,0.007555,0.000840,0.003730,0.000929,10.0,scale,linear,"{'clf__C': 10, 'clf__gamma': 'scale', 'clf__ke...",1.000000,0.958333,...,0.966667,0.031180,5,0.979167,0.958333,0.968750,0.989583,1.000000,0.979167,0.014731
9,0.008423,0.001159,0.005934,0.004206,10.0,scale,rbf,"{'clf__C': 10, 'clf__gamma': 'scale', 'clf__ke...",0.958333,0.958333,...,0.966667,0.016667,5,0.989583,0.979167,0.968750,0.989583,1.000000,0.985417,0.010623
10,0.007676,0.000981,0.004175,0.001297,10.0,auto,linear,"{'clf__C': 10, 'clf__gamma': 'auto', 'clf__ker...",1.000000,0.958333,...,0.966667,0.031180,5,0.979167,0.958333,0.968750,0.989583,1.000000,0.979167,0.014731
11,0.010279,0.002308,0.003729,0.000228,10.0,auto,rbf,"{'clf__C': 10, 'clf__gamma': 'auto', 'clf__ker...",0.958333,0.958333,...,0.966667,0.016667,5,0.989583,0.979167,0.968750,0.989583,1.000000,0.985417,0.010623


In [28]:
print(grid_clf.best_params_)

{'clf__C': 0.1, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}


In [29]:
svc_final = SVC(C = 0.1, gamma = 'scale', kernel = 'linear', random_state=42)

In [30]:
svc_final.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [31]:
pred = svc_final.predict(X_test)

In [32]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



연습
1. boston 데이터셋을 로드
2. 독립변수, 종속변수로 데이터를 분할
3. train, test로 8:2 비율로 변환
4. KFold을 이용하여 10개의 폴드를 생성 (shuffle True)
5. 파이프라인 생성 ( StandardScaler(), SVR() )
6. 파라미터 조합을 생성 ( svr모델에서 C ( 1, 10, 100 ), kernel ( 'linear', 'rbf' ) , epsilon ( 0.1, 0.2, 0.5 ) )
7. GridSearchCV 베스트 모델 선정의 기준은 neg_mean_squared_error
8. 최고의 조합을 찾은 뒤 R2-score를 이용하여 성능을 확인

In [33]:
boston = pd.read_csv("../csv/boston.csv")

In [34]:
x = boston.drop('Price', axis=1)
y = boston['Price']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [36]:
kfold = KFold(n_splits=10, shuffle = True, random_state=42)

In [37]:
pipe_reg = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('reg', SVR())
    ]
)

In [38]:
params_reg = {
    "reg__C" : [1, 10, 100], 
    'reg__kernel' : ['linear', 'rbf'], 
    "reg__epsilon" : [0.1, 0.2, 0.5]
}

In [39]:
grid_reg = GridSearchCV(
    estimator= pipe_reg, 
    param_grid= params_reg, 
    scoring= 'neg_mean_squared_error', 
    cv = kfold, 
    verbose=1, 
    refit = True
)

In [40]:
grid_reg.fit(X_train, y_train)

Fitting 10 folds for each of 18 candidates, totalling 180 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...reg', SVR())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'reg__C': [1, 10, ...], 'reg__epsilon': [0.1, 0.2, ...], 'reg__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for ea

In [41]:
# 최적의 모델이 저장되어있는 속성 : best_estimator_ 
grid_reg.best_estimator_.score(X_test, y_test)

0.8354920503211445

In [42]:
grid_reg.score(X_test, y_test)

-12.063990309898506

In [43]:
from sklearn.metrics import r2_score

In [44]:

pipe_pred = grid_reg.best_estimator_.predict(X_test)
grid_pred = grid_reg.predict(X_test)

In [45]:
print(r2_score(y_test, pipe_pred))
print(r2_score(y_test, grid_pred))

0.8354920503211445
0.8354920503211445
